# 감정 분류 모델 학습 (KcELECTRA) — v2 vs v3 비교

원본 노트북의 인코딩 디버깅 셀, 중복 전처리 시도(v1, 사람문장1만 확인용), 실행되지 않은 죽은 코드 셀은 제거했습니다.
v1(6클래스, class weight 없음)은 test accuracy 0.48로 v2보다 성능이 낮아 이 버전에서는 제외했습니다.
v3 평가 시 이전 버전의 `trainer` 변수를 잘못 재사용하던 버그를 수정해 v2와 v3를 동일한 test set으로 정확히 비교합니다.

## 0. 환경 설정

In [ ]:
!pip install transformers datasets evaluate accelerate -q

import numpy as np
import torch
import pandas as pd
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, EarlyStoppingCallback)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss
import evaluate

MODEL_NAME = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=48)

def to_ds(df):
    d = Dataset.from_pandas(df[['text', 'label_id']].rename(columns={'label_id': 'label'}))
    return d.map(tokenize, batched=True)

acc_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1_macro': f1_metric.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

## 1. 데이터 로드 및 전처리

`사람문장1`만 사용하고, `상처` 라벨을 `슬픔`으로 병합한 v2/v3 공통 전처리입니다.

In [ ]:
df = pd.read_csv('sample_data/emotion_data.csv', encoding='cp949')
print(df.shape)

In [ ]:
sub = df[['사람문장1', '감정_대분류']].dropna()
sub.columns = ['text', 'label']
train_df = sub.drop_duplicates().reset_index(drop=True)

merge_map = {'기쁨': '기쁨', '당황': '당황', '분노': '분노', '불안': '불안', '상처': '슬픔', '슬픔': '슬픔'}
train_df['label'] = train_df['label'].map(merge_map)

print(train_df.shape)
print(train_df['label'].value_counts())

In [ ]:
le = LabelEncoder()
train_df['label_id'] = le.fit_transform(train_df['label'])
num_labels = len(le.classes_)
print(dict(enumerate(le.classes_)))  # 5개여야 정상

train_data, temp = train_test_split(train_df, test_size=0.2, stratify=train_df['label_id'], random_state=42)
val_data, test_data = train_test_split(temp, test_size=0.5, stratify=temp['label_id'], random_state=42)

train_ds, val_ds, test_ds = to_ds(train_data), to_ds(val_data), to_ds(test_data)

## 2. v2 — class weight 미적용

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

args_v2 = TrainingArguments(
    output_dir='./emotion_model_v2',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    warmup_steps=0.1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=8,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=torch.cuda.is_available(),
)

trainer_v2 = Trainer(
    model=model, args=args_v2,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_v2.train()

In [ ]:
best_ckpt_v2 = trainer_v2.state.best_model_checkpoint
print('v2 best checkpoint:', best_ckpt_v2)

model_v2_best = AutoModelForSequenceClassification.from_pretrained(best_ckpt_v2)
eval_args = TrainingArguments(output_dir='./tmp_eval_v2', per_device_eval_batch_size=64, report_to='none')
trainer_v2_eval = Trainer(model=model_v2_best, args=eval_args)

preds_v2 = trainer_v2_eval.predict(test_ds)
y_pred_v2 = preds_v2.predictions.argmax(axis=-1)
report_v2 = classification_report(test_data['label_id'], y_pred_v2, target_names=le.classes_, output_dict=True)
print(classification_report(test_data['label_id'], y_pred_v2, target_names=le.classes_))

## 3. v3 — class weight 적용 (Weighted CrossEntropy)

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data['label_id']),
    y=train_data['label_id']
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print(dict(zip(le.classes_, class_weights.tolist())))

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

args_v3 = TrainingArguments(
    output_dir='./emotion_model_v3',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    warmup_steps=0.1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=8,
    weight_decay=0.05,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=torch.cuda.is_available(),
)

trainer_v3 = WeightedTrainer(
    model=model, args=args_v3,
    train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer_v3.train()

In [ ]:
# 주의: 원본 노트북에서는 이 평가 셀이 'trainer'(v1 객체)를 잘못 참조해
# v3 성능이 실제로 측정된 적이 없었습니다. trainer_v3로 수정했습니다.
best_ckpt_v3 = trainer_v3.state.best_model_checkpoint
print('v3 best checkpoint:', best_ckpt_v3)

model_v3_best = AutoModelForSequenceClassification.from_pretrained(best_ckpt_v3)
eval_args = TrainingArguments(output_dir='./tmp_eval_v3', per_device_eval_batch_size=64, report_to='none')
trainer_v3_eval = Trainer(model=model_v3_best, args=eval_args)

preds_v3 = trainer_v3_eval.predict(test_ds)
y_pred_v3 = preds_v3.predictions.argmax(axis=-1)
report_v3 = classification_report(test_data['label_id'], y_pred_v3, target_names=le.classes_, output_dict=True)
print(classification_report(test_data['label_id'], y_pred_v3, target_names=le.classes_))

## 4. v2 vs v3 비교

In [ ]:
print(f"v2  accuracy={report_v2['accuracy']:.4f}  macro f1={report_v2['macro avg']['f1-score']:.4f}")
print(f"v3  accuracy={report_v3['accuracy']:.4f}  macro f1={report_v3['macro avg']['f1-score']:.4f}")